# Integrated Machine Learning Application

This project integrates four different machine learning modules into a single menu-driven application:

1. **Regression Analysis** – Comparison of Linear, Ridge, PCR, PLS, and MLP models
2. **Clustering Analysis** – K-Means with optimal K selection via Silhouette Score
3. **Text Sentiment Analysis** – SimpleRNN on IMDB movie reviews
4. **Image Classification** – CNN on Fashion-MNIST

Upon completion of each module, results are appended to `results.txt`.

## **Datasets:**
- `proje_ev_fiyatlari.csv` – 400 houses, 5 features + price
- `proje_musteri_segmentasyonu.csv` – 300 customers, income + spending score
- `imdb_dataset.csv` – 50,000 movie reviews (Kaggle: IMDB Dataset of 50K Movie Reviews)
- Fashion-MNIST – Built-in via Keras (28×28 grayscale, 10 classes)


In [9]:
from datetime import datetime
from time import sleep

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import clear_output
from sklearn.cluster import KMeans
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from tabulate import tabulate
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.layers import (
    Conv2D,
    Dense,
    Embedding,
    Flatten,
    Input,
    MaxPooling2D,
    SimpleRNN,
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

print("All libraries loaded successfully ✅")

All libraries loaded successfully ✅


In [10]:
# Results File Utility Functions
def initialize_results_file() -> None:
    """Create results.txt from scratch, removing any previous content."""
    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write("Integrated Machine Learning Application - Results\n")
        f.write("=" * 60 + "\n")
        f.write(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 60 + "\n")


def save_section_header(section_name: str) -> None:
    """Append a section header to results.txt."""
    with open(RESULTS_FILE, "a", encoding="utf-8") as f:
        f.write(f"\n{'=' * 60}\n{section_name}\n{'=' * 60}\n")


def save_result(section: str, method: str, metric_name: str, value) -> None:
    """Append a single metric line to results.txt."""
    with open(RESULTS_FILE, "a", encoding="utf-8") as f:
        f.write(f"[{section}] {method} | {metric_name}: {value}\n")


In [11]:
# Constants
RESULTS_FILE = "results.txt"

# IMDB dataset column names
IMDB_REVIEW_COL = "review"
IMDB_SENTIMENT_COL = "sentiment"


# Results File Utility Functions
def initialize_results_file() -> None:
    """Create results.txt from scratch, removing any previous content."""
    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write("Integrated Machine Learning Application - Results\n")
        f.write("=" * 60 + "\n")
        f.write(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 60 + "\n")


def save_section_header(section_name: str) -> None:
    """Append a section header to results.txt."""
    with open(RESULTS_FILE, "a", encoding="utf-8") as f:
        f.write(f"\n{'=' * 60}\n{section_name}\n{'=' * 60}\n")


def save_result(section: str, method: str, metric_name: str, value) -> None:
    """Append a single metric line to results.txt."""
    with open(RESULTS_FILE, "a", encoding="utf-8") as f:
        f.write(f"[{section}] {method} | {metric_name}: {value}\n")


# Load Datasets
data_prices = pd.read_csv("proje_ev_fiyatlari.csv")
data_customer = pd.read_csv("proje_musteri_segmentasyonu.csv")
data_imdb = pd.read_csv("imbd_dataset.csv")

print("Datasets loaded successfully ✅")
print(
    f"  House prices       : {data_prices.shape[0]} rows × {data_prices.shape[1]} cols"
)
print(
    f"  Customer segments  : {data_customer.shape[0]} rows × {data_customer.shape[1]} cols"
)
print(f"  IMDB reviews       : {data_imdb.shape[0]} rows × {data_imdb.shape[1]} cols")
print(f"\nIMDB columns: {list(data_imdb.columns)}")
print(f"Sentiment distribution:\n{data_imdb[IMDB_SENTIMENT_COL].value_counts()}")

# Initialize results file (clear previous content)
initialize_results_file()
print(f"Results file '{RESULTS_FILE}' initialized ✅")

Datasets loaded successfully ✅
  House prices       : 400 rows × 6 cols
  Customer segments  : 300 rows × 2 cols
  IMDB reviews       : 50000 rows × 2 cols

IMDB columns: ['review', 'sentiment']
Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64
Results file 'results.txt' initialized ✅


## Section 1 – Regression Analysis

Housing price prediction using 5 features (area, room_count, building_age, distance_to_center, floor) from 400 houses.

**Models:** Linear Regression, Ridge, PCR, PLS, MLP (Artificial Neural Network)

**Preprocessing:** `StandardScaler` is applied to all models for feature standardization. Train/test split (`test_size=0.2, random_state=42`) is shared across all models to ensure fair comparison.

**Metrics:** MSE and R²

In [12]:
# Section 1: Regression Analysis


def prepare_regression_data(data):
    """
    Common data preparation for regression.
    - Single train/test split shared across all models
    - StandardScaler for feature standardization
    """

    # Data Preparation
    X = data.drop("fiyat", axis=1)
    y = data["fiyat"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test


def run_regression(data, components=3):

    X_train, X_test, y_train, y_test = prepare_regression_data(data)

    results = []

    # 1. Linear Regression
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append(
        [
            "Linear Regression",
            round(mean_squared_error(y_test, y_pred), 2),
            round(r2_score(y_test, y_pred), 3),
        ]
    )

    # 2. Ridge Regression
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append(
        [
            "Ridge",
            round(mean_squared_error(y_test, y_pred), 2),
            round(r2_score(y_test, y_pred), 3),
        ]
    )

    # 3. PCR (Principal Component Regression)
    pca = PCA(n_components=components)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)
    results.append(
        [
            f"PCR ({components} components)",
            round(mean_squared_error(y_test, y_pred), 2),
            round(r2_score(y_test, y_pred), 3),
        ]
    )

    # 4. PLS (Partial Least Squares)
    model = PLSRegression(n_components=components)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append(
        [
            f"PLS ({components} components)",
            round(mean_squared_error(y_test, y_pred), 2),
            round(r2_score(y_test, y_pred), 3),
        ]
    )

    # 5. Artificial Neural Network (MLP)
    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))

    model_mlp = MLPRegressor(
        hidden_layer_sizes=(64,),
        max_iter=10000,
        random_state=42,
    )
    model_mlp.fit(X_train, y_train_scaled.ravel())

    y_pred_scaled = model_mlp.predict(X_test)
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))
    results.append(
        [
            "Artificial Neural Network (MLP)",
            round(mean_squared_error(y_test, y_pred), 2),
            round(r2_score(y_test, y_pred), 3),
        ]
    )

    return results


## Section 2 – Clustering Analysis

K-Means clustering on 300 customers using annual income and spending score features.

**Optimal K selection:** The K value that maximizes the Silhouette Score is selected. Elbow plot is provided for visualization.

**Preprocessing:** `StandardScaler` is mandatory; income (tens of thousands TL) and spending score (1–100) are on vastly different scales.

In [13]:
# Section 2: Clustering Analysis


def run_clustering(data):
    """
    K-Means clustering analysis.
    Optimal K is determined by maximizing the Silhouette Score.
    Returns (optimal_k, silhouette_score).ınput?
    """

    # Data Prepation
    X = data[["yillik_gelir", "harcama_puani"]]

    # Standardization
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Analyze across K values
    k_values = list(range(1, 11))
    inertia_list = []
    silhouette_list = []

    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=36, n_init=10)
        kmeans.fit(X_scaled)
        inertia_list.append(kmeans.inertia_)

        if k == 1:
            silhouette_list.append(None)
        else:
            silhouette_list.append(silhouette_score(X_scaled, kmeans.labels_))

    # Print summary table
    table = []
    for k in k_values:
        sil = silhouette_list[k - 1]
        table.append(
            [
                k,
                round(inertia_list[k - 1], 2),
                round(sil, 2) if sil is not None else "-",
            ]
        )

    print("Inertia and Silhouette Scores across K values:")
    print(
        tabulate(
            table,
            headers=["K", "Inertia", "Silhouette"],
            tablefmt="simple",
            numalign="center",
        )
    )

    # Optimal K: Maximum Silhouette Score
    valid_k = [k for k in k_values if k > 1]
    valid_sil = [s for s in silhouette_list if s is not None]
    optimal_idx = int(np.argmax(valid_sil))
    k_opt = valid_k[optimal_idx]
    sil_opt = valid_sil[optimal_idx]

    print(f"\nOptimal K = {k_opt} (Silhouette = {sil_opt:.2f})")

    # Elbow & Silhouette Plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(k_values, inertia_list, linestyle="--", marker="o", color="steelblue")
    axes[0].axvline(x=k_opt, color="red", linestyle=":", label=f"Optimal K = {k_opt}")
    axes[0].set_xlabel("Number of Clusters (K)")
    axes[0].set_ylabel("Inertia (WCSS)")
    axes[0].set_title("Elbow Method")
    axes[0].set_xticks(k_values)
    axes[0].legend()
    axes[0].grid(True, linestyle=":", alpha=0.5)

    axes[1].plot(valid_k, valid_sil, linestyle="--", marker="o", color="coral")
    axes[1].axvline(x=k_opt, color="red", linestyle=":", label=f"Optimal K = {k_opt}")
    axes[1].set_xlabel("Number of Clusters (K)")
    axes[1].set_ylabel("Silhouette Score")
    axes[1].set_title("Silhouette Score vs K")
    axes[1].set_xticks(valid_k)
    axes[1].legend()
    axes[1].grid(True, linestyle=":", alpha=0.5)

    plt.tight_layout()
    fig.savefig("clustering_elbow_silhouette.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Final Clustering with Optimal K
    kmeans_opt = KMeans(n_clusters=k_opt, random_state=36, n_init=10)
    cluster = kmeans_opt.fit(X_scaled)
    labels = cluster.labels_
    centers = cluster.cluster_centers_

    # Inverse transform for visualization
    X_original = scaler.inverse_transform(X_scaled)
    centers_original = scaler.inverse_transform(centers)

    df = pd.DataFrame(X_original, columns=["Annual Income", "Spending Score"])
    df["Cluster"] = labels

    fig2, ax2 = plt.subplots(figsize=(12, 6))
    sns.scatterplot(
        ax=ax2,
        data=df,
        x="Annual Income",
        y="Spending Score",
        hue="Cluster",
        palette="bright",
        s=50,
        alpha=0.7,
    )

    ax2.scatter(
        centers_original[:, 0],
        centers_original[:, 1],
        c="black",
        s=120,
        alpha=0.6,
        marker="X",
        label="Cluster Centers",
    )

    ax2.legend(loc="right")
    ax2.set_xlabel("Annual Income")
    ax2.set_ylabel("Spending Score")
    ax2.grid(True, linestyle=":", alpha=0.5)
    ax2.set_xticks(np.arange(0, max(df["Annual Income"]) + 20, 10))
    ax2.set_yticks(np.arange(0, max(df["Spending Score"]) + 10, 10))
    fig2.savefig("clustering_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()

    return k_opt, round(sil_opt, 2)


## Section 3 – Text Sentiment Analysis (SimpleRNN)

Binary sentiment classification on IMDB movie reviews.

**Dataset:** Kaggle – "IMDB Dataset of 50K Movie Reviews" (`imdb_dataset.csv`)

**Architecture:** Embedding → SimpleRNN → Dense(sigmoid)

**Preprocessing:** Texts are converted to numerical sequences via Tokenizer and padded to uniform length with `pad_sequences`.

In [14]:
# Section 3: Text Sentiment Analysis (SimpleRNN)


def run_sentiment(
    data,
    max_words=10000,
    max_len=200,
    embedding_dim=64,
    rnn_units=64,
    epochs=5,
    batch_size=64,
):
    """
    SimpleRNN-based sentiment analysis on IMDB reviews.
    Returns test accuracy.
    """

    # Data Preparation
    reviews = data[IMDB_REVIEW_COL].astype(str).values
    sentiments = (data[IMDB_SENTIMENT_COL].str.lower() == "positive").astype(int).values

    X_train_text, X_test_text, y_train, y_test = train_test_split(
        reviews, sentiments, test_size=0.2, random_state=42, stratify=sentiments
    )

    print(f"Train: {len(X_train_text)} | Test: {len(X_test_text)}")

    # Tokenizer
    tokenizer = Tokenizer(num_words=max_words)
    tokenizer.fit_on_texts(X_train_text)

    X_train_seq = tokenizer.texts_to_sequences(X_train_text)
    X_test_seq = tokenizer.texts_to_sequences(X_test_text)

    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post")
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post")

    # Model
    model = Sequential(
        [
            Input(shape=(max_len,)),
            Embedding(max_words, embedding_dim),
            SimpleRNN(rnn_units),
            Dense(1, activation="sigmoid"),
        ]
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    model.summary()

    # Training
    model.fit(
        X_train_pad,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_test_pad, y_test),
        verbose=1,
    )

    # Test Accuracy
    test_loss, test_acc = model.evaluate(X_test_pad, y_test, verbose=0)
    print(f"\nTest Loss:     {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

    return round(float(test_acc), 4), round(float(test_loss), 4)


## Section 4 – Image Classification (CNN)

CNN-based classification on Fashion-MNIST (28×28 grayscale, 10 classes).

**Dataset:** Built-in via Keras (`tensorflow.keras.datasets.fashion_mnist`)

**Architecture:** Conv2D → MaxPooling → Conv2D → MaxPooling → Flatten → Dense → Dense(softmax)

**Preprocessing:** Pixel values normalized to 0–1 range (/255).

In [15]:
# Section 4: Image Classification (CNN)

FASHION_LABELS = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]


def run_image_classification(epochs=10, batch_size=64):
    """
    CNN-based Fashion-MNIST classification.
    Returns test accuracy.
    """

    # Load Data
    (X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

    # Normalize and reshape
    X_train = X_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
    X_test = X_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

    print(f"Train: {X_train.shape} | Test: {X_test.shape}")

    # Model Training
    model = Sequential(
        [
            Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
            MaxPooling2D((2, 2)),
            Conv2D(64, (3, 3), activation="relu"),
            MaxPooling2D((2, 2)),
            Flatten(),
            Dense(128, activation="relu"),
            Dense(10, activation="softmax"),
        ]
    )

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    model.summary()

    # Model Training
    model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_test, y_test),
        verbose=1,
    )

    # Test Accuracy
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest Accuracy: {test_acc:.4f}")

    return round(float(test_acc), 4)


## Menu

User selects an analysis from the menu, the corresponding module runs, and results are saved to `results.txt`. The menu loops until the user chooses to exit.

In [17]:
# Menu Loop


def print_menu() -> None:
    """Display the main menu."""
    print("""
    ╔══════════════════════════════════════════════════╗
    ║        Integrated Machine Learning               ║
    ║              Application                         ║
    ║                                                  ║
    ║   1. Regression Analysis                         ║
    ║   2. Clustering Analysis                         ║
    ║   3. Text Sentiment Analysis                     ║
    ║   4. Image Classification                        ║
    ║   5. Exit                                        ║
    ║                                                  ║
    ╚══════════════════════════════════════════════════╝
    """)


def run_menu() -> None:
    """Main menu loop. Results are appended to results.txt."""
    while True:
        print_menu()

        try:
            choice = int(input("Your choice (1-5): ").strip())

            # Section 1: Regression
            if choice == 1:
                print("\n─── Regression Analysis ───\n")

                try:
                    n_comp = int(
                        input(
                            "Number of components for PCR and PLS [default: 3]: "
                        ).strip()
                        or "3"
                    )
                except ValueError:
                    n_comp = 3

                results = run_regression(data_prices, n_comp)

                print("\nModel Comparison:")
                print(
                    tabulate(
                        results,
                        headers=["Model", "MSE", "R²"],
                        tablefmt="simple",
                        numalign="center",
                    )
                )

                # Best model
                df_results = pd.DataFrame(results, columns=["Model", "MSE", "R²"])
                best = df_results.sort_values("R²", ascending=False).iloc[0]
                print(
                    f"\nBest model: {best['Model']} (R²={best['R²']}, MSE={best['MSE']})"
                )

                # Save to results.txt
                save_section_header("Section 1 - Regression Analysis")
                for model_name, mse, r2 in results:
                    save_result("Regression", model_name, "MSE", mse)
                    save_result("Regression", model_name, "R²", r2)
                print(f"\nResults saved to '{RESULTS_FILE}' ✅")

            # Section 2: Clustering
            elif choice == 2:
                print("\n─── Clustering Analysis ───\n")
                k_opt, sil_score = run_clustering(data_customer)

                save_section_header("Section 2 - Clustering Analysis")
                save_result("Clustering", "K-Means", "Optimal K", k_opt)
                save_result("Clustering", "K-Means", "Silhouette Score", sil_score)
                save_result(
                    "Clustering",
                    "K-Means",
                    "Elbow & Silhouette Plot",
                    "clustering_elbow_silhouette.png",
                )
                save_result(
                    "Clustering",
                    "K-Means",
                    "Cluster Scatter Plot",
                    "clustering_scatter.png",
                )
                print(f"\nResults saved to '{RESULTS_FILE}' ✅")

            # Section 3: Sentiment Analysis
            elif choice == 3:
                print("\n─── Text Sentiment Analysis ───\n")
                test_acc, test_loss = run_sentiment(data_imdb)

                save_section_header("Section 3 - Text Sentiment Analysis")
                save_result(
                    "Sentiment Analysis", "SimpleRNN", "Test Accuracy", test_acc
                )
                save_result("Sentiment Analysis", "SimpleRNN", "Test Loss", test_loss)
                print(f"\nResults saved to '{RESULTS_FILE}' ✅")

            # Section 4: Image Classification
            elif choice == 4:
                print("\n─── Image Classification ───\n")
                test_acc = run_image_classification()

                save_section_header("Section 4 - Image Classification")
                save_result("Image Classification", "CNN", "Test Accuracy", test_acc)
                print(f"\nResults saved to '{RESULTS_FILE}' ✅")

            # Exit
            elif choice == 5:
                print("\nExiting program...")
                break

            else:
                print("Invalid choice. Please enter a number between 1 and 5.")

        except ValueError:
            print("Invalid input. Please enter a number.")

        except KeyboardInterrupt:
            print("\nInterrupted by user.")
            break


run_menu()



    ╔══════════════════════════════════════════════════╗
    ║        Integrated Machine Learning               ║
    ║              Application                         ║
    ║                                                  ║
    ║   1. Regression Analysis                         ║
    ║   2. Clustering Analysis                         ║
    ║   3. Text Sentiment Analysis                     ║
    ║   4. Image Classification                        ║
    ║   5. Exit                                        ║
    ║                                                  ║
    ╚══════════════════════════════════════════════════╝
    

─── Regression Analysis ───


Model Comparison:
Model                              MSE     R²
-------------------------------  -------  -----
Linear Regression                43795.6  0.911
Ridge                            43846.3  0.911
PCR (3 components)               138874   0.719
PLS (3 components)               50872.8  0.897
Artificial Neural Network (MLP) 